<a href="https://colab.research.google.com/github/lelange/tni_lab/blob/main/colab_registration_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MRI Registration Workshop (TNI lab)

We start with installing a few required packages and importing them into our workspace.

In [ ]:
# Install packages (locally, a virtual environmend is recommended). In colab, you can do this directly.
!pip -q install antspyx antspynet nilearn nibabel matplotlib scipy ipywidgets

In [ ]:
import os
import io
import math
import time
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.optimize import minimize
from skimage import data, transform
from skimage.color import rgb2gray
from skimage.util import img_as_float
from IPython.display import display, Markdown
import ipywidgets as widgets

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import ants
from nilearn import datasets, plotting, image
from nilearn.plotting import view_img, plot_anat, plot_roi

import antspynet

# 0) A simple 2D registration exercise

In [ ]:
# @title
# Build a toy 2D registration example.
# We create one fixed image and one moving image that starts misaligned.

base = img_as_float(rgb2gray(data.astronaut()))
base = transform.resize(base, (256, 256), anti_aliasing=True)

fixed_2d = base

# Create a deliberately misaligned version.
moving_2d_start = ndimage.rotate(base, angle=18, reshape=False, order=1)
moving_2d_start = ndimage.shift(moving_2d_start, shift=(18, -14), order=1)

def transform_2d(img, tx=0, ty=0, angle=0):
    rotated = ndimage.rotate(img, angle=angle, reshape=False, order=1)
    shifted = ndimage.shift(rotated, shift=(ty, tx), order=1)
    return shifted

def mse(a, b):
    return float(np.mean((a - b) ** 2))

def joint_hist_mi(a, b, bins=32):
    # Simple mutual information estimate from a joint histogram.
    hist_2d, _, _ = np.histogram2d(a.ravel(), b.ravel(), bins=bins)
    pxy = hist_2d / np.sum(hist_2d)
    px = np.sum(pxy, axis=1)
    py = np.sum(pxy, axis=0)
    px_py = np.outer(px, py)
    nz = pxy > 0
    return float(np.sum(pxy[nz] * np.log(pxy[nz] / px_py[nz])))

def show_registration_2d(tx=0, ty=0, angle=0.0):
    moved = transform_2d(moving_2d_start, tx=tx, ty=ty, angle=angle)
    current_mse = mse(fixed_2d, moved)
    current_mi = joint_hist_mi(fixed_2d, moved)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(fixed_2d, cmap="gray")
    axes[0].set_title("Fixed image")
    axes[0].axis("off")

    axes[1].imshow(moved, cmap="gray")
    axes[1].set_title("Moving image after transform")
    axes[1].axis("off")

    axes[2].imshow(fixed_2d, cmap="gray", alpha=0.6)
    axes[2].imshow(moved, cmap="magma", alpha=0.4)
    axes[2].set_title(f"Overlay\nMSE = {current_mse:.4f}, MI = {current_mi:.4f}")
    axes[2].axis("off")
    plt.show()

tx_slider = widgets.IntSlider(value=0, min=-40, max=40, step=1, description="tx")
ty_slider = widgets.IntSlider(value=0, min=-40, max=40, step=1, description="ty")
angle_slider = widgets.FloatSlider(value=0.0, min=-30, max=30, step=0.5, description="angle")

ui = widgets.VBox([tx_slider, ty_slider, angle_slider])
out = widgets.interactive_output(
    show_registration_2d,
    {"tx": tx_slider, "ty": ty_slider, "angle": angle_slider}
)

display(Markdown("**Try to move the sliders until the images line up better.**"))
display(ui, out)

## 1) Proceeding to brain images: Upload a T1 NIfTI

In [ ]:
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded. Please upload one T1 NIfTI (.nii or .nii.gz).")

t1_path = list(uploaded.keys())[0]
print("Uploaded:", t1_path)

## 2) Load template and atlas from nilearn

We use:
- **MNI152 T1 template** from `nilearn.datasets.load_mni152_template`
- **Harvard-Oxford cortical atlas** from `nilearn.datasets.fetch_atlas_harvard_oxford`

The Nilearn MNI template is the asymmetrical ICBM152 2009 release re-sampled by Nilearn.

In [ ]:
atlas_choice = "Harvard-Oxford (cortical)"  #@param ["Harvard-Oxford (cortical)","Juelich","Destrieux"]
atlas_folder = "atlas"
atlas_path = "atlas/atlas.nii.gz"

template_img = datasets.load_mni152_template(resolution=1)

if atlas_choice == "Harvard-Oxford (cortical)":
    atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-1mm')

elif atlas_choice == "Juelich":
    atlas = datasets.fetch_atlas_juelich("maxprob-thr0-1mm")

elif atlas_choice == "Destrieux":
    atlas = datasets.fetch_atlas_destrieux_2009(data_dir=atlas_folder)
    atlas_path = os.path.join(atlas_folder, atlas.maps)

template_path = "mni_template.nii.gz"

atlas_labels = atlas.labels

#save template and atlas to read in later
nib.save(template_img, template_path)
if type(atlas.maps) != str:
  nib.save(atlas.maps, atlas_path)

#print("Template:", template_path)
#print("Atlas:", atlas_path)
print("Number of atlas labels:", len(atlas_labels))

In [ ]:
# Quick visual sanity check
plotting.plot_anat(template_img, title="MNI152 template", display_mode="ortho")
plt.show()

plotting.plot_roi(atlas.maps, bg_img=template_img, title=f"{atlas_choice} atlas on template")
plt.show()

## 3) Helper functions

These functions:
- read NIfTI into ANTs,
- normalize intensities for easier visual comparison,
- compute simple error maps,
- show mid-slice comparisons.

In [ ]:
# @title
def ants_from_nifti(path):
    return ants.image_read(path)

def nib_from_ants(img, out_path=None):
    # Reconstruct affine from ANTs image properties
    direction = img.direction
    spacing = img.spacing
    origin = img.origin

    affine = np.eye(4)
    affine[:3, :3] = direction @ np.diag(spacing)
    affine[:3, 3] = origin

    nii = nib.Nifti1Image(img.numpy(), affine)
    if out_path is not None:
        nib.save(nii, out_path)
    return nii

def nib_to_ants(nib_img):
    data = nib_img.get_fdata()
    affine = nib_img.affine

    spacing = np.sqrt((affine[:3, :3] ** 2).sum(0))
    origin = affine[:3, 3]
    direction = affine[:3, :3] / spacing

    return ants.from_numpy(
        data,
        spacing=tuple(spacing),
        origin=tuple(origin),
        direction=direction
    )

def robust_minmax(x, low=1, high=99):
    a = np.asarray(x, dtype=np.float32)
    lo, hi = np.percentile(a[np.isfinite(a)], [low, high])
    if hi <= lo:
        hi = lo + 1e-6
    b = np.clip((a - lo) / (hi - lo), 0, 1)
    return b

def normalized_abs_diff(fixed_arr, moved_arr):
    f = robust_minmax(fixed_arr)
    m = robust_minmax(moved_arr)
    return np.abs(f - m)

def mse(fixed_arr, moved_arr):
    f = robust_minmax(fixed_arr)
    m = robust_minmax(moved_arr)
    return float(np.mean((f - m) ** 2))

def corrcoef_img(fixed_arr, moved_arr):
    f = robust_minmax(fixed_arr).ravel()
    m = robust_minmax(moved_arr).ravel()
    mask = np.isfinite(f) & np.isfinite(m)
    if mask.sum() < 10:
        return np.nan
    return float(np.corrcoef(f[mask], m[mask])[0, 1])

def show_registration_summary(fixed_arr, moved_arr, title_prefix=""):
    diff = normalized_abs_diff(fixed_arr, moved_arr)
    z = fixed_arr.shape[2] // 2
    y = fixed_arr.shape[1] // 2
    x = fixed_arr.shape[0] // 2

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    planes = [
        (slice(None), slice(None), z, "Axial"),
        (slice(None), y, slice(None), "Coronal"),
        (x, slice(None), slice(None), "Sagittal"),
    ]

    for row, (sx, sy, sz, plane_name) in enumerate(planes):
        axes[row, 0].imshow(np.rot90(fixed_arr[sx, sy, sz]), cmap="gray")
        axes[row, 0].set_title(f"{plane_name}: fixed/template")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(np.rot90(moved_arr[sx, sy, sz]), cmap="gray")
        axes[row, 1].set_title(f"{plane_name}: moved T1")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(np.rot90(diff[sx, sy, sz]), cmap="hot")
        axes[row, 2].set_title(f"{plane_name}: abs diff")
        axes[row, 2].axis("off")

    fig.suptitle(
        f"{title_prefix}  |  MSE={mse(fixed_arr, moved_arr):.4f}, "
        f"corr={corrcoef_img(fixed_arr, moved_arr):.4f}",
        fontsize=14
    )
    plt.tight_layout()
    plt.show()



## 4) Load subject T1 and inspect initial mismatch

In [ ]:
t1_ants = ants.image_read(t1_path)

# Brain extraction for T1
prob_mask = antspynet.brain_extraction(t1_ants, modality="t1")

# Convert probability mask to binary mask
brain_mask = ants.threshold_image(prob_mask, 0.6, 1.0)

# Apply mask
subject_t1 = t1_ants * brain_mask

In [ ]:

template_ants = nib_to_ants(template_img) #ants.image_read(template_path)

show_registration_summary(
    template_ants.numpy(),
    ants.resample_image_to_target(subject_t1, template_ants).numpy(),
    title_prefix="Before registration (resampled only)"
)


## 5) Run different registration methods

We test three different registration types:
- **Rigid** = move the brain without changing its shape
- **Affine** = global reshaping: scale, shear, rotate, translate
- **Non-linear (SyN)** = localized warping possible

This can take a few minutes, especially the non-linear one.

In [ ]:
import time
timings = {}
# Rigid
start = time.perf_counter()
rigid_reg = ants.registration(
    fixed=template_ants,
    moving=subject_t1,
    type_of_transform="Rigid"
)
timings["Rigid"] = time.perf_counter() - start
print(f"Rigid done in {timings['Rigid']:.2f} s")

# Affine
start = time.perf_counter()
affine_reg = ants.registration(
    fixed=template_ants,
    moving=subject_t1,
    type_of_transform="Affine"
)
timings["Affine"] = time.perf_counter() - start
print(f"Affine done in {timings['Affine']:.2f} s")

# SyN (Nonlinear)
start = time.perf_counter()
syn_reg = ants.registration(
    fixed=template_ants,
    moving=subject_t1,
    type_of_transform="SyN"
)
timings["Nonlinear_SyN"] = time.perf_counter() - start
print(f"SyN done in {timings['Nonlinear_SyN']:.2f} s")

print("\nFinished all registrations.\n")

### While registration runs, think about this:


What would you inspect, when you later inspect the results:
1. Which registration run will be the fastest and why?
2. Which of the Affine transformations will improve size/proportion matching?
3. Which anatomical details of the brain could be better matched by non-linear registration than with Rigid or Affine?

*(double click in the cell to write down some notes)*

### Still running? How about solving a quiz?

A short quiz to test your understanding of transformations to be used in registration of brain MRI. You can check the results as soon as the registration process above has finished :)

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

quiz_items = [
    {
        "question": "1. Which transformation allows only translation and rotation?",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Rigid",
        "explanation": "Rigid transformations preserve shape and size. They only rotate and translate the image."
    },
    {
        "question": "2. Which transformation can additionally scale and shear an image?",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Affine",
        "explanation": "Affine transformations include rotation, translation, scaling, and shearing."
    },
    {
        "question": "3. Which transformation is best suited to match subtle anatomical differences between two different brains?",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Non-linear",
        "explanation": "Non-linear registration can locally deform the image to match anatomical variability across subjects."
    },
    {
        "question": "4. If you want to align a subject's head position to a template without changing its shape, which transform is most appropriate?",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Rigid",
        "explanation": "Rigid is the right first step when only pose differences should be corrected."
    },
    {
        "question": "5. Why might affine registration still be insufficient for inter-subject registration?",
        "options": [
            "Because affine is too slow",
            "Because affine cannot model local anatomical differences",
            "Because affine cannot rotate images"
        ],
        "answer": "Because affine cannot model local anatomical differences",
        "explanation": "Affine changes global geometry only. It cannot warp local structures to match subject-specific anatomy."
    },
    {
        "question": "6. Which transformation has the highest risk of overfitting or introducing unrealistic warping if used carelessly?",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Non-linear",
        "explanation": "Non-linear registration is powerful, but it can distort anatomy if the settings or data quality are poor."
    },
    {
        "question": "7. In a typical T1-to-template workflow, which order is most sensible?",
        "options": [
            "Non-linear → Affine → Rigid",
            "Rigid → Affine → Non-linear",
            "Affine → Non-linear → Rigid"
        ],
        "answer": "Rigid → Affine → Non-linear",
        "explanation": "Registration usually progresses from coarse global alignment to finer local alignment."
    },
    {
        "question": "8. If an atlas is mapped back to subject space, what interpolation should usually be used for the atlas labels?",
        "options": ["Linear interpolation", "Nearest-neighbor interpolation", "Gaussian smoothing"],
        "answer": "Nearest-neighbor interpolation",
        "explanation": "Atlas labels are discrete regions, so nearest-neighbor preserves the label identities."
    },
]

title = widgets.HTML(
    value="""
    <h2>Registration Quiz</h2>
    <p>Solve these questions while the registration is running.</p>
    <p><b>Goal:</b> understand when to use rigid, affine, and non-linear transforms.</p>
    """
)

question_boxes = []
feedback_outputs = []

for item in quiz_items:
    q = widgets.HTML(value=f"<b>{item['question']}</b>")
    choices = widgets.RadioButtons(
        options=item["options"],
        value=None,
        layout=widgets.Layout(width='auto')
    )
    feedback = widgets.Output()
    question_boxes.append((item, q, choices, feedback))
    feedback_outputs.append(feedback)

submit_button = widgets.Button(
    description="Check answers",
    button_style="success",
    icon="check"
)

reset_button = widgets.Button(
    description="Reset quiz",
    button_style="warning",
    icon="refresh"
)

score_output = widgets.Output()

def on_submit_clicked(b):
    with score_output:
        clear_output()
        score = 0
        total = len(question_boxes)

        for item, q, choices, feedback in question_boxes:
            with feedback:
                clear_output()
                if choices.value is None:
                    display(Markdown("⚠️ No answer selected yet."))
                elif choices.value == item["answer"]:
                    score += 1
                    display(Markdown(f"✅ **Correct.** {item['explanation']}"))
                else:
                    display(Markdown(
                        f"❌ **Not quite.** Correct answer: **{item['answer']}**.  \n{item['explanation']}"
                    ))

        display(Markdown(f"## Score: **{score}/{total}**"))

        if score == total:
            display(Markdown("Excellent. You are ready to justify which registration result is best."))
        elif score >= total * 0.7:
            display(Markdown("Good. Review the explanations, especially where local vs. global deformation matters."))
        else:
            display(Markdown("Review the differences between global alignment and local warping before interpreting results."))

def on_reset_clicked(b):
    for item, q, choices, feedback in question_boxes:
        choices.value = None
        with feedback:
            clear_output()
    with score_output:
        clear_output()

submit_button.on_click(on_submit_clicked)
reset_button.on_click(on_reset_clicked)

display(title)

for item, q, choices, feedback in question_boxes:
    display(widgets.VBox([q, choices, feedback]))
    display(widgets.HTML("<hr>"))

display(widgets.HBox([submit_button, reset_button]))
display(score_output)



In [ ]:
# @title
scenario_title = widgets.HTML(
    value="<h3>Scenario decision quiz</h3><p>Choose the best transform for each situation:</p>"
)

scenario_items = [
    {
        "question": "A subject moved slightly in the scanner, and you want to align one structural scan to another from the same session.",
        "options": ["Non-linear", "Rigid", "Affine"],
        "answer": "Rigid",
        "explanation": "For same-subject scans with mainly pose differences, rigid is often sufficient."
    },
    {
        "question": "You want to align a subject T1 to MNI template and account for differences in brain size and global proportions.",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Affine",
        "explanation": "Affine adds scaling and shearing, which helps with global anatomical differences."
    },
    {
        "question": "You want the best anatomical correspondence between cortical and subcortical structures across two different people.",
        "options": ["Rigid", "Affine", "Non-linear"],
        "answer": "Non-linear",
        "explanation": "Non-linear registration is needed for local shape differences across individuals."
    },
]

scenario_widgets = []
scenario_score_output = widgets.Output()
scenario_submit = widgets.Button(description="Check scenario answers", button_style="info")

for item in scenario_items:
    q = widgets.HTML(value=f"<b>{item['question']}</b>")
    choices = widgets.RadioButtons(options=item["options"], value=None)
    out = widgets.Output()
    scenario_widgets.append((item, q, choices, out))

def on_scenario_submit(b):
    with scenario_score_output:
        clear_output()
        score = 0
        total = len(scenario_widgets)
        for item, q, choices, out in scenario_widgets:
            with out:
                clear_output()
                if choices.value == item["answer"]:
                    score += 1
                    display(Markdown(f"✅ **Correct.** {item['explanation']}"))
                else:
                    display(Markdown(f"❌ Correct answer: **{item['answer']}**. {item['explanation']}"))
        display(Markdown(f"### Scenario score: **{score}/{total}**"))

scenario_submit.on_click(on_scenario_submit)

display(scenario_title)
for item, q, choices, out in scenario_widgets:
    display(widgets.VBox([q, choices, out]))
    display(widgets.HTML("<hr>"))
display(scenario_submit)
display(scenario_score_output)

## 6) Compare outputs and error maps

In [ ]:
results = {
    "Rigid": rigid_reg["warpedmovout"],
    "Affine": affine_reg["warpedmovout"],
    "Nonlinear_SyN": syn_reg["warpedmovout"],
}

fixed_arr = template_ants.numpy()

metrics = []
for name, moved_img in results.items():
    moved_arr = moved_img.numpy()
    metrics.append({
        "method": name,
        "mse": mse(fixed_arr, moved_arr),
        "corr": corrcoef_img(fixed_arr, moved_arr),
    })


for row in metrics:
    print(row)

In [ ]:
for name, moved_img in results.items():
    show_registration_summary(
        template_ants.numpy(),
        moved_img.numpy(),
        title_prefix=f"{name} registration"
    )

## 7) Interactive inspection in 2D

These viewers are often more useful than scalar metrics.

Inspect:
- ventricles
- cortical outline
- cerebellum
- frontal and occipital poles
- whether tissue boundaries line up consistently

The student should decide which registration is best overall.

In [ ]:
# Save transformed images to NIfTI so nilearn's interactive viewer can use them easily
out_dir = "registration_outputs"
os.makedirs(out_dir, exist_ok=True)

moved_paths = {}
for name, moved_img in results.items():
    path = os.path.join(out_dir, f"{name}_in_template_space.nii.gz")
    #ants.image_write(moved_img, path)
    nib_from_ants(moved_img, path)
    #nib.save(nib_from_ants(moved_img), path)
    moved_paths[name] = path
    print(name, "->", path)

In [ ]:
for name, path in moved_paths.items():
    print(f"Interactive viewer: {name}")
    print(path)
    #display(view_img(path, bg_img=template_path, opacity=0.55, title=f"{name} on template"))
    img = image.load_img(path)

    view = view_img(
        img,
        bg_img=template_img,
        opacity=0.55,
        title=f"{name} on template",
        cmap="gray"
    )
    display(view)

## 8) Student choice: select the best registration

Set `best_method` after inspecting the overlays and error maps.

In [ ]:
best_method = "Rigid"  #@param ["Rigid", "Affine", "Nonlinear_SyN"]
assert best_method in results
best_reg = {
    "Rigid": rigid_reg,
    "Affine": affine_reg,
    "Nonlinear_SyN": syn_reg,
}[best_method]

print("Selected best method:", best_method)

## 9) Warp the atlas from template space into subject T1 space

Important:
- Atlas is a **label image**
- We therefore use **nearest-neighbor interpolation**
- Because registration was estimated from subject -> template,
  we apply the **inverse transform** to move template atlas -> subject T1

In [ ]:
atlas_ants = nib_to_ants(atlas.maps) #ants.image_read(atlas_path) #ants_from_nifti(atlas_path)

atlas_in_subject = ants.apply_transforms(
    fixed=subject_t1,
    moving=atlas_ants,
    transformlist=best_reg["invtransforms"],
    interpolator="nearestNeighbor"
)

atlas_in_subject_path = os.path.join(out_dir, f"atlas_in_subject_{best_method}.nii.gz")
#nib.save(nib_from_ants(atlas_in_subject), atlas_in_subject_path)
#ants.image_write(atlas_in_subject, atlas_in_subject_path)
nib_from_ants(atlas_in_subject, atlas_in_subject_path)

subject_t1_path = os.path.join(out_dir, "subject_t1_copy.nii.gz")
#nib.save(nib_from_ants(subject_t1), subject_t1_path)
#ants.image_write(subject_t1, subject_t1_path)
nib_from_ants(subject_t1, subject_t1_path)

print("Atlas in subject space:", atlas_in_subject_path)
print("Subject T1 copy:", subject_t1_path)

## 10) Visualize atlas regions on the subject brain

In [ ]:
plotting.plot_roi(
    atlas_in_subject_path,
    bg_img=subject_t1_path,
    title=f"Atlas on subject T1 ({best_method})",
    display_mode="ortho",
    draw_cross=False
)
plt.show()

In [ ]:
ants.plot(
    subject_t1,
    atlas_in_subject)

In [ ]:
display(view_img(atlas_in_subject_path, bg_img=subject_t1_path, opacity=0.45,
                 title=f"Atlas warped to subject T1 ({best_method})", cmap="tab20"))

## 11) Optional: inspect a few named atlas labels

This cell helps students connect region names to the atlas label numbers.

In [ ]:
for i, label in enumerate(atlas_labels[:25]):
    print(f"{i:3d}: {label}")

## 12) Optional: isolate one atlas region in subject space

Set a region index below, then visualize only that region.

In [ ]:
region_index = 1  # change this after checking atlas_labels above

atlas_subj_nii = nib.load(atlas_in_subject_path)
atlas_subj_data = atlas_subj_nii.get_fdata()

region_mask_data = (atlas_subj_data == region_index).astype(np.uint8)
region_mask_img = nib.Nifti1Image(region_mask_data, atlas_subj_nii.affine, atlas_subj_nii.header)

region_mask_path = os.path.join(out_dir, f"region_{region_index:03d}_mask.nii.gz")
nib.save(region_mask_img, region_mask_path)

print("Selected region:", atlas_labels[region_index] if region_index < len(atlas_labels) else "unknown")
display(view_img(region_mask_path, bg_img=subject_t1_path, opacity=0.6, cmap="tab20",
                 title=f"Region {region_index}: {atlas_labels[region_index] if region_index < len(atlas_labels) else 'unknown'}"))

## 14) Reflection questions

1. Which registration looked best, and why?
2. Where did rigid registration fail?
3. What improved with affine registration?
4. What changed further with the non-linear transform?
5. Did the best scalar metric match the best visual impression?
6. In a real study, what quality checks would you want?